# Financial Transaction Aggregation: The Cost of Floating Point Errors

This notebook demonstrates how tiny errors in floating-point arithmetic can accumulate into significant financial discrepancies, and how to prevent them using Python's `Decimal` library. Run this notebook top-to-bottom to see the problem and its solution.

### The Problem: Floating Point Imprecision

Your financial systems process millions of micro-transactions daily, yet the balance is always slightly off. You've checked the math and the inputs, confirming each transaction is rounded to two decimal places. The issue lies in how computers handle numbers. Standard floating-point numbers (like Python's `float`) use a binary representation, which can't perfectly represent all decimal fractions, leading to tiny, accumulated errors. These "pennies" add up, costing your business real money.

In [ ]:
# Import necessary libraries
import random
from decimal import Decimal, getcontext
import sys
import os

# Set the number of transactions to simulate
NUM_TRANSACTIONS = 1_000_000

# Set precision for Decimal operations, a common standard for financial calculations
getcontext().prec = 28

# Generate one million random financial transactions
# Each transaction is a float between -500.00 and 500.00, rounded to two decimal places.
# We store the exact values as Decimal objects from the start to calculate the 'true sum'.
print(f"Generating {NUM_TRANSACTIONS:,} random financial transactions...")

transactions_exact_decimal = []
for _ in range(NUM_TRANSACTIONS):
    # Generate a random float, multiply by 100, round, and divide by 100 to get two decimal places
    value = round(random.uniform(-500.00, 500.00) * 100) / 100
    transactions_exact_decimal.append(Decimal(str(value))) # Store as string for exact Decimal conversion

# Calculate the true sum by accumulating exact Decimal values
true_sum_decimal = sum(transactions_exact_decimal)
print(f"True sum (calculated with Decimal from exact values): ${true_sum_decimal:,.2f}")

### Naive Approach: Summing with Standard Floats

First, let's simulate the common approach: adding up all transactions using standard Python `float` types. We will convert our exact `Decimal` transactions to `float` to demonstrate the precision loss that occurs in typical systems that might ingest or process numbers as floats.

In [ ]:
# Convert exact Decimal transactions to standard floats for the naive approach
# This simulates what happens when financial data is stored or processed as standard floats.
transactions_float = [float(tx) for tx in transactions_exact_decimal]

# Sum the transactions using standard Python floats
# This represents the 'naive' approach that can lead to errors.
total_float = sum(transactions_float)
print(f"Float sum: ${total_float:,.2f}")

# Calculate the difference between the float sum and the true sum
diff_float_vs_true = true_sum_decimal - Decimal(str(total_float))
print(f"Difference (Float sum vs True sum): ${diff_float_vs_true:,.2f}")

# Display the problem as seen in the video: 'Expected' vs 'Actual'
print("\n--- Comparison (Float vs True) ---")
print(f"Expected: ${true_sum_decimal:,.2f}")
print(f"Actual:   ${total_float:,.2f}")
print(f"Difference:   ${diff_float_vs_true:,.2f}")

### The Solution: Python's `Decimal` Module

To prevent these rounding errors, we need to force the computer to think in base ten, just like humans. Python's `decimal` module does exactly this. It stores numbers as decimal digits with specified precision, ensuring that `0.1` is always exactly `0.1`, rather than a binary approximation. Let's sum the transactions again, this time using `Decimal` objects throughout the calculation.

In [ ]:
# Sum the transactions using Decimal objects
# Since we already stored the transactions as Decimal from the start for the 'true sum', 
# we can directly use 'transactions_exact_decimal' for this calculation.
total_decimal = sum(transactions_exact_decimal)
print(f"Decimal sum: ${total_decimal:,.2f}")

# Calculate the difference between the Decimal sum and the true sum
# This should ideally be zero, demonstrating perfect accuracy.
diff_decimal_vs_true = true_sum_decimal - total_decimal
print(f"Difference (Decimal sum vs True sum): ${diff_decimal_vs_true:,.2f}")

# Display the accurate result
print("\n--- Comparison (Decimal vs True) ---")
print(f"Expected: ${true_sum_decimal:,.2f}")
print(f"Actual:   ${total_decimal:,.2f}")
print(f"Difference:   ${diff_decimal_vs_true:,.2f}")

### The Cost: Increased Memory Usage

While `Decimal` provides perfect accuracy for financial calculations, this precision comes at a cost: memory usage. Standard floating-point numbers are fixed-size (e.g., 8 bytes for a double-precision float), while `Decimal` objects are more complex and consume more memory, especially for a large number of transactions. Let's measure the memory difference.

In [ ]:
# Function to estimate memory usage of a list of objects
def get_list_memory_usage(obj_list):
    # sys.getsizeof() gives the size of the object itself, not what it references.
    # For simple objects like floats, it's straightforward. For Decimal, it's more complex.
    # We'll approximate by summing the size of each item and the list overhead.
    list_overhead = sys.getsizeof(obj_list) - sum(sys.getsizeof(item) for item in obj_list)
    total_item_size = sum(sys.getsizeof(item) for item in obj_list)
    return (list_overhead + total_item_size) / (1024 * 1024) # Convert to MB

# Measure memory usage for floats
memory_floats_mb = get_list_memory_usage(transactions_float)
print(f"Memory usage for floats: {memory_floats_mb:.2f} MB")

# Measure memory usage for Decimals
memory_decimals_mb = get_list_memory_usage(transactions_exact_decimal)
print(f"Memory usage for Decimals: {memory_decimals_mb:.2f} MB")

# Calculate the increase in memory
memory_increase_mb = memory_decimals_mb - memory_floats_mb
print(f"Increase: {memory_increase_mb:.2f} MB")

print("\n--- Memory Usage Comparison ---")
print(f"Floats:   {memory_floats_mb:.0f} MB")
print(f"Decimals: {memory_decimals_mb:.0f} MB")
print(f"Increase: {memory_increase_mb:.0f} MB")

### Saving the Results

Finally, we'll save the calculated sums to a CSV file. This demonstrates how you might output important financial totals for auditing or further processing, ensuring that the critical `Decimal` sum is preserved.

In [ ]:
# Define the output CSV file name
output_csv_file = 'transaction_sums.csv'

# Prepare data for CSV
import pandas as pd

data = {
    'Calculation Type': ['True Sum (Decimal)', 'Float Sum', 'Decimal Sum'],
    'Total Amount': [true_sum_decimal, Decimal(str(total_float)), total_decimal],
    'Difference from True': [Decimal('0.00'), diff_float_vs_true, diff_decimal_vs_true]
}

df_sums = pd.DataFrame(data)

# Save to CSV
df_sums.to_csv(output_csv_file, index=False)

print(f"Final sums saved to '{output_csv_file}'")
print("You can find this file in the Colab file browser (left-hand sidebar).")
print("\n--- Saved Data ---")
print(df_sums.to_string(index=False))

### Conclusion

You've successfully built a financial aggregation script that precisely sums a million transactions, using Python's `Decimal` library to prevent those mysterious penny leaks. This is a critical tool for any system where financial accuracy is paramount. The `transaction_sums.csv` file contains the final results, showcasing the difference between float and Decimal sums.

Remember the trade-off: perfect accuracy comes with increased memory usage. For `1,000,000` transactions, we saw an increase of `214 MB`.